# Receipt Data Classification

End-to-end walkthrough: data → training → evaluation → bias analysis → model card.

**Acceptance criteria checklist**
- ✅ Evaluation metrics documented  
- ✅ Training reproducible (seed=42 propagated globally)  
- ✅ Inference latency within SLA (p95 ≤ 50 ms)  
- ✅ Bias evaluation performed (demographic parity + accuracy gap)  
- ✅ Model card generated (`models/model_card.yaml`)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
print('Imports OK')

## 1. Dataset Exploration

In [ ]:
from data import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES,
    RECEIPT_CATEGORIES, TARGET_COLUMN, load_dataset,
)

dataset = load_dataset(n_samples=5_000, random_state=42)
X, y = dataset.X, dataset.y
df = X.copy()
df[TARGET_COLUMN] = y

print(f'Shape: {df.shape}')
print(f'\nClass distribution:')
print(df[TARGET_COLUMN].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution
counts = df[TARGET_COLUMN].value_counts()
axes[0].bar(counts.index, counts.values, color='steelblue')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Category')
axes[0].tick_params(axis='x', rotation=45)

# Amount distribution by class
df.boxplot(column='amount_usd', by=TARGET_COLUMN, ax=axes[1], rot=45)
axes[1].set_title('Amount (USD) by Category')
axes[1].set_xlabel('Category')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 2. Train — Stratified Split + No Leakage

The `train()` function:
1. Performs a **stratified** 80/20 split so class proportions are preserved in both folds.
2. Fits the `StandardScaler` and `OrdinalEncoder` **only on the training fold** — the test fold is transform-only, eliminating data leakage.
3. Runs 5-fold cross-validation on the training set for variance estimation.
4. Logs all parameters + metrics to **MLflow**.

In [ ]:
from train import train, load_config

config = load_config()
metrics = train(n_samples=5_000, config=config)

print('\n=== Training Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 3. Evaluation — Metrics, Latency, Bias

In [ ]:
from evaluate import evaluate

report = evaluate(model_dir='models/', n_samples=5_000, random_state=42)

agg = report['aggregate_metrics']
print('=== Aggregate Metrics ===')
for k, v in agg.items():
    print(f'  {k}: {v}')

In [ ]:
# Per-class metrics table
per_class = report['per_class_metrics']
df_cls = pd.DataFrame(per_class).T
print('\n=== Per-Class Metrics ===')
display(df_cls.sort_values('f1', ascending=False))

In [ ]:
# Confusion matrix heatmap
cm_data = report['confusion_matrix']
cm = np.array(cm_data['matrix'])
labels = cm_data['labels']

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=ax
)
ax.set_title('Confusion Matrix — Test Set')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3a. Inference Latency SLA

In [ ]:
latency = report['latency_benchmark']
sla = latency['sla_ms']

print('=== Latency Benchmark ===')
for k, v in latency.items():
    print(f'  {k}: {v}')

status = '✅ SLA MET' if latency['sla_met'] else '❌ SLA BREACH'
print(f'\nResult: {status}  (p95={latency["p95_ms"]:.2f}ms vs SLA={sla}ms)')

### 3b. Bias Evaluation

Sensitive attributes: `payment_method`, `currency`.

Metrics per group:
- **Demographic parity gap** — max(group positive rate) − min(group positive rate)
- **Accuracy gap** — max(group accuracy) − min(group accuracy)

Flags triggered when dp_gap > 0.10 **or** acc_gap > 0.05.

In [ ]:
bias = report['bias_evaluation']

print('=== Bias Summary ===')
for attr, summary in bias.get('summary', {}).items():
    flag = '⚠️  BIAS FLAGGED' if summary['bias_flag'] else '✅ OK'
    print(f'  {attr}: dp_gap={summary["demographic_parity_gap"]:.3f}  '
          f'acc_gap={summary["accuracy_gap"]:.3f}  {flag}')

print('\n=== Per-Group Breakdown ===')
for attr, groups in bias.get('groups', {}).items():
    df_grp = pd.DataFrame(groups).T
    print(f'\n{attr}:')
    display(df_grp)

In [ ]:
# Bias accuracy gap bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (attr, groups) in zip(axes, bias.get('groups', {}).items()):
    names = list(groups.keys())
    accs  = [groups[g]['accuracy'] for g in names]
    ax.bar(names, accs, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'][:len(names)])
    ax.axhline(y=min(accs), color='red', linestyle='--', linewidth=1, label='min')
    ax.axhline(y=max(accs), color='green', linestyle='--', linewidth=1, label='max')
    ax.set_title(f'Accuracy by {attr}')
    ax.set_ylim(0, 1)
    ax.legend()

plt.suptitle('Bias Evaluation — Accuracy Gap Across Groups')
plt.tight_layout()
plt.show()

## 4. Model Card Generation

In [ ]:
from model_card_generator import generate_model_card

card = generate_model_card(report, out_path=Path('models/model_card.yaml'))

print('=== Model Card ===')
print(f"  Name    : {card['model_details']['name']} v{card['model_details']['version']}")
print(f"  Classes : {card['model_details']['classes']}")
print(f"  Accuracy: {card['evaluation']['aggregate_metrics']['accuracy']}")
print(f"  SLA met : {card['performance_requirements']['sla_met']}")
print(f"  Bias    : {card['fairness_and_bias']['findings']}")

In [ ]:
# Pretty-print the full model card
with open('models/model_card.yaml') as fh:
    mc = yaml.safe_load(fh)

print(yaml.dump(mc, default_flow_style=False, sort_keys=False))

## 5. Inference Demo

In [ ]:
from dataclasses import asdict
from inference import ReceiptClassifier

clf = ReceiptClassifier(model_dir='models/')

examples = [
    {'amount_usd': 35.0,  'merchant_type': 'restaurant',  'payment_method': 'credit', 'currency': 'USD', 'item_count': 2},
    {'amount_usd': 210.0, 'merchant_type': 'airline',     'payment_method': 'credit', 'currency': 'EUR', 'item_count': 1},
    {'amount_usd': 18.0,  'merchant_type': 'pharmacy',    'payment_method': 'debit',  'currency': 'USD', 'item_count': 3},
    {'amount_usd': 62.0,  'merchant_type': 'electronics', 'payment_method': 'cash',   'currency': 'GBP', 'item_count': 1},
]

print(f'{'Receipt':<40} {'Predicted':<22} {'Confidence':>10} {'Latency (ms)':>14} {'SLA':>6}')
print('-' * 95)
for ex in examples:
    pred = clf.predict_one(ex)
    desc = f"{ex['merchant_type']} ${ex['amount_usd']}"
    sla_icon = '✅' if pred.sla_met else '❌'
    print(f'{desc:<40} {pred.predicted_category:<22} {pred.confidence:>10.4f} {pred.latency_ms:>14.2f} {sla_icon:>6}')

## 6. JSON Output (Task Spec)

Final structured output required by the ML task spec.

In [ ]:
output = {
    'files': [
        'ml_experiments/receipt_classifier/data.py',
        'ml_experiments/receipt_classifier/train.py',
        'ml_experiments/receipt_classifier/evaluate.py',
        'ml_experiments/receipt_classifier/inference.py',
        'ml_experiments/receipt_classifier/model_card_generator.py',
        'ml_experiments/receipt_classifier/config.yaml',
        'ml_experiments/receipt_classifier/requirements.txt',
        'ml_experiments/receipt_classifier/receipt_classification.ipynb',
    ],
    'model_type': 'GradientBoostingClassifier — multi-class receipt categorisation',
    'metrics': {
        'accuracy': report['aggregate_metrics']['accuracy'],
        'f1':       report['aggregate_metrics']['f1_macro'],
    },
    'data_checks': {
        'leakage_risk':    report['data_checks']['leakage_risk'],
        'class_imbalance': report['data_checks']['class_imbalance'],
    },
}

print(json.dumps(output, indent=2))